# Capítulo 8 · Red Neuronal Cuántica (QNN)

## Objetivos

1. Comprender la arquitectura de una red neuronal cuántica (QNN) basada en circuitos variacionales.
2. Implementar un clasificador cuántico de dos clases usando `SamplerQNN` de Qiskit.
3. Entrenar la red con gradiente basado en `parameter-shift rule`.
4. Comparar el rendimiento con una red neuronal clásica equivalente.

---

## 8B.1 Arquitectura QNN

Una QNN combina:

- **Capa de embedding**: Circuito $U(\mathbf{x})$ que codifica los datos clásicos en el estado cuántico.
- **Capa variacional**: Circuito $W(\boldsymbol{\theta})$ con parámetros entrenables.
- **Medida**: El valor esperado de un observable sirve como salida de la red.

La regla de desplazamiento de parámetro (parameter-shift rule) permite calcular gradientes exactos:

$$\frac{\partial E}{\partial \theta_i} = \frac{E(\theta_i + \pi/2) - E(\theta_i - \pi/2)}{2}$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.neural_networks import SamplerQNN, EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_algorithms.optimizers import COBYLA, SPSA

print('QNN módulos cargados.')

In [ ]:
# ── Dataset: círculos concéntricos ───────────────────────────────
np.random.seed(0)
X, y = make_circles(n_samples=80, noise=0.1, factor=0.5)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)

# Mapear etiquetas a {-1, +1} para la QNN
y_train_qnn = 2 * y_train - 1
y_test_qnn  = 2 * y_test - 1

print(f'Dataset: {len(X_train)} train, {len(X_test)} test')

# Visualización
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[y==0, 0], X[y==0, 1], c='#58a6ff', label='Clase 0')
ax.scatter(X[y==1, 0], X[y==1, 1], c='#f78166', label='Clase 1')
ax.set_title('Dataset: círculos concéntricos')
ax.legend()
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

In [ ]:
# ── Circuito QNN ──────────────────────────────────────────────────
n_qubits = 2

feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=1)
ansatz      = RealAmplitudes(n_qubits, reps=2, entanglement='linear')

# Circuito completo: embedding + variacional
qnn_circuit = QuantumCircuit(n_qubits)
qnn_circuit.compose(feature_map, inplace=True)
qnn_circuit.compose(ansatz,      inplace=True)

print('Circuito QNN:')
print(qnn_circuit.decompose().draw('text'))

print(f'\nParámetros de entrada (datos): {feature_map.num_parameters}')
print(f'Parámetros entrenables:        {ansatz.num_parameters}')

In [ ]:
# ── Definir y entrenar la QNN ─────────────────────────────────────
from qiskit.quantum_info import SparsePauliOp

observable = SparsePauliOp.from_list([('ZI', 1.0)])

estimator_qnn = EstimatorQNN(
    circuit=qnn_circuit,
    observables=observable,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
)

# Clasificador cuántico
loss_history = []

def callback(weights, obj_val):
    loss_history.append(obj_val)

qnn_classifier = NeuralNetworkClassifier(
    neural_network=estimator_qnn,
    optimizer=COBYLA(maxiter=150),
    callback=callback,
)

print('Entrenando la QNN…')
qnn_classifier.fit(X_train, y_train_qnn)
print('Entrenamiento completado.')

# Evaluación
y_pred = qnn_classifier.predict(X_test)
acc = accuracy_score(y_test_qnn, y_pred)
print(f'\nExactitud QNN: {acc:.4f}')

In [ ]:
# Curva de aprendizaje
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history, color='#58a6ff', linewidth=1.5)
ax.set_xlabel('Iteración')
ax.set_ylabel('Pérdida')
ax.set_title('Curva de aprendizaje — QNN clasificador')
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 8B.2 Ejercicios propuestos

1. Sustituye el optimizador `COBYLA` por `ADAM` e implementa la regla de desplazamiento de parámetros manualmente para estimar los gradientes.

2. Aumenta el número de capas del ansatz. ¿Cómo evoluciona la exactitud y el tiempo de entrenamiento?

3. Investiga el problema del **barren plateau** en QNNs profundas: ¿por qué el gradiente tiende a cero con el número de qubits y capas?